# PoS tagging italiano con BERT XXL, ELECTRA XXL e GePpeTto



In [1]:
%pip install -q "transformers==4.57.1" "datasets==4.8.5" "accelerate==1.11.0" "safetensors>=0.5,<1" "numpy>=2,<3" "scikit-learn==1.8.0" "pandas>=2,<4"


Note: you may need to restart the kernel to use updated packages.


## 2. Configuration


In [ ]:
import csv, gc, hashlib, importlib.metadata, json, math, platform, re, statistics, time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, cast
# Load pandas/PyArrow before PyTorch to avoid the Windows native import conflict.
from datasets import Dataset, load_dataset
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset as TorchDataset
from IPython.display import display
from transformers import (AutoConfig, AutoModel, AutoTokenizer, GPT2ForTokenClassification,
                          PreTrainedModel, Trainer, TrainingArguments, set_seed)
from transformers.modeling_outputs import TokenClassifierOutput

SEEDS = [42, 123, 456]
VARIANTS = ["original", "lowercase"]
UPOS_TAGS = ("ADJ", "ADP", "ADV", "AUX", "CCONJ", "DET", "INTJ", "NOUN", "NUM",
             "PART", "PRON", "PROPN", "PUNCT", "SCONJ", "SYM", "VERB", "X")
LABEL_IDS = {tag: i for i, tag in enumerate(UPOS_TAGS)}

DATA = {
    "repo_id": "universal-dependencies/universal_dependencies",
    "config": "it_isdt",
    "ud_version": "2.15",
    "release_tag": "ud2.15-loader-v2.3.0",
    "revision": "8426077f2e778f20d46a5b89a187e4f991cfe6c4",
    "prepared_dir": "data/prepared",
    "parquet_sha256": {
        "train": "7cf4df5e42e54335c1a649a248cb3d2aa85876f22a4ae1c32b60202a30d2543d",
        "dev": "9a15c35ba8d307c9d623eddacff6f759a25d7f52ed575f2bbe2d796216ef6e1f",
        "test": "ce4b6b696417174934c8a39da6bb88e1894689e8fa1bb2b87fe108a123a1bc15"
    }
}

MODELS = {
    "bert": {
        "kind": "encoder",
        "repo_id": "dbmdz/bert-base-italian-xxl-cased",
        "revision": "9bdf84aedf00aa4e99b01a6f2c76f6070ac7240b",
        "epochs": 10,
        "learning_rate": 2e-05,
        "batch_size": 8,
        "eval_batch_size": 16,
        "gradient_accumulation": 1,
        "max_length": 512,
        "dropout": 0.1,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1
    },
    "electra": {
        "kind": "encoder",
        "repo_id": "dbmdz/electra-base-italian-xxl-cased-discriminator",
        "revision": "c6d38915b86f51db0ba5d2a2c150b01c2289c8e7",
        "epochs": 10,
        "learning_rate": 2e-05,
        "batch_size": 8,
        "eval_batch_size": 16,
        "gradient_accumulation": 1,
        "max_length": 512,
        "dropout": 0.1,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1
    },
    "geppetto": {
        "kind": "decoder",
        "tagger_version": "gpt2_upos_mean_pooling_v1",
        "repo_id": "LorenzoDeMattei/GePpeTto",
        "revision": "d4b79e7cb2b41925adf2837a7715243f56eaf705",
        "epochs": 10,
        "learning_rate": 2e-05,
        "batch_size": 8,
        "eval_batch_size": 16,
        "gradient_accumulation": 1,
        "max_length": 512,
        "dropout": 0.1,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1
    }
}

PROJECT = Path.cwd().resolve()
for candidate in (PROJECT, PROJECT.parent, PROJECT / "ai-language-model-evaluation"):
    if (candidate / "notebooks/Italian_UPOS_AI.ipynb").is_file():
        PROJECT = candidate
        break
else:
    raise RuntimeError("Aprire il notebook dalla nuova cartella del progetto o da notebooks/")

EXPERIMENT = {
    "schema": "upos_mean_pooling_trainer_v1", "dataset": DATA,
    "models": MODELS, "seeds": SEEDS, "variants": VARIANTS,
    "pooling": "mean", "selection_metric": "upos_macro_f1_all",
    "output_dir": "outputs/upos_trainer_v1",
}
OUTPUT_DIR = PROJECT / EXPERIMENT["output_dir"]

EXPERIMENT_DIR = OUTPUT_DIR
SYSTEMS = {f"{model}_{variant}": {
    **spec, "model": model, "variant": variant, "name": f"{model.upper()} / {variant}",
} for model, spec in MODELS.items() for variant in VARIANTS}
ACTIVE_SYSTEMS = list(SYSTEMS)
EXPERIMENT["active_systems"] = ACTIVE_SYSTEMS
EXPERIMENT["trainer"] = {"eval_strategy": "epoch", "save_strategy": "epoch",
    "load_best_model_at_end": True, "save_total_limit": 1, "disable_tqdm": False}
print("Device:", torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU")


## 3. Data


In [3]:
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def require_revision(revision):
    if not re.fullmatch(r"[0-9a-f]{40}", revision):
        raise ValueError("Use an immutable 40-character Hugging Face commit SHA")


def transform_tokens(tokens, variant):
    if variant == "original":
        return list(tokens)
    if variant == "lowercase":
        return [word.lower() for word in tokens]
    raise ValueError(f"Unknown input variant: {variant}")


def normalize_sentence(row, index, label_names=None):
    tokens = list(row["tokens"])
    tags = list(row["upos"])
    if label_names is not None:
        tags = [label_names[tag] if isinstance(tag, int) else tag for tag in tags]
    if not tokens or len(tokens) != len(tags):
        raise ValueError(f"Invalid word/tag alignment in sentence {index}")
    if any(not isinstance(word, str) or not word for word in tokens):
        raise ValueError(f"Invalid word form in sentence {index}")
    if set(tags) - set(UPOS_TAGS):
        raise ValueError(f"Non-UPOS annotation in sentence {index}; require syntactic words only")
    return {"sent_id": str(row.get("sent_id", row.get("idx", index))), "tokens": tokens, "upos": tags}


def prepare_corpus(config, project, local_parquet=None):

    spec = config["dataset"]
    require_revision(spec["revision"])
    target = Path(project) / spec["prepared_dir"]
    manifest_path = target / "manifest.json"
    if manifest_path.exists():
        load_corpus(config, project)
        return json.loads(manifest_path.read_text(encoding="utf-8"))
    if target.exists() and any(target.iterdir()):
        raise RuntimeError("Incomplete prepared corpus: use a new prepared_dir or restore its files")
    if local_parquet is None:
        dataset = load_dataset(spec["repo_id"], spec["config"], revision=spec["revision"], cache_dir=str(Path(project) / "data/hf-cache"))
    else:
        local_parquet = Path(local_parquet)
        if not local_parquet.is_absolute():
            local_parquet = Path(project) / local_parquet
        files = {split: local_parquet / f"{split}.parquet" for split in ("train", "dev", "test")}
        expected = spec.get("parquet_sha256", {})
        for split, path in files.items():
            if split not in expected or sha256(path) != expected[split]:
                raise ValueError(f"Local Parquet does not match the pinned Hugging Face snapshot: {split}")
        dataset = load_dataset("parquet", data_files={split: str(path) for split, path in files.items()}, cache_dir=str(Path(project) / "data/hf-cache"))
    split_names = {"train": "train", "dev": "dev" if "dev" in dataset else "validation", "test": "test"}
    corpus = {}
    for split, source in split_names.items():
        table = dataset[source]
        feature = table.features["upos"]
        label_names = getattr(getattr(feature, "feature", None), "names", None)
        corpus[split] = [normalize_sentence(row, i, label_names) for i, row in enumerate(table)]
    if set(tag for row in corpus["train"] for tag in row["upos"]) != set(UPOS_TAGS):
        raise ValueError("Training corpus does not match the expected 17-tag UPOS inventory")
    target.mkdir(parents=True, exist_ok=True)
    manifest = {"dataset": spec, "labels": list(UPOS_TAGS), "splits": {}}
    for split, rows in corpus.items():
        path = target / f"{split}.jsonl"
        path.write_text("".join(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\n" for row in rows), encoding="utf-8")
        manifest["splits"][split] = {"sentences": len(rows), "words": sum(len(row["tokens"]) for row in rows), "sha256": sha256(path)}
    manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    return manifest


def load_corpus(config, project):
    target = Path(project) / config["dataset"]["prepared_dir"]
    manifest = json.loads((target / "manifest.json").read_text(encoding="utf-8"))
    if manifest["dataset"] != config["dataset"] or manifest["labels"] != list(UPOS_TAGS):
        raise ValueError("Prepared corpus and configuration differ; use another prepared_dir")
    corpus = {}
    for split, details in manifest["splits"].items():
        path = target / f"{split}.jsonl"
        if sha256(path) != details["sha256"]:
            raise ValueError(f"Prepared corpus hash mismatch: {split}")
        corpus[split] = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
        if len(corpus[split]) != details["sentences"] or sum(len(row["tokens"]) for row in corpus[split]) != details["words"]:
            raise ValueError(f"Prepared corpus count mismatch: {split}")
    return corpus, manifest


local_parquet = PROJECT / "data/hf-parquet"
local_available = all((local_parquet / f"{split}.parquet").is_file() for split in ("train", "dev", "test"))
prepare_corpus(EXPERIMENT, PROJECT, local_parquet if local_available else None)
sentences, data_manifest = load_corpus(EXPERIMENT, PROJECT)
for split, details in data_manifest["splits"].items():
    print(f"{split}: {details['sentences']:,} frasi, {details['words']:,} parole")

display(pd.DataFrame([{"Split": split, "Sentences": len(rows),
    "Words": sum(len(row["tokens"]) for row in rows)} for split, rows in sentences.items()]))


train: 13,121 frasi, 276,014 parole
dev: 564 frasi, 11,907 parole
test: 482 frasi, 10,417 parole


,Split,Sentences,Words
0,train,13121,276014
1,dev,564,11907
2,test,482,10417


## 4. Tokenization

All models predict one of the 17 UPOS labels per gold word. Mean pooling combines the contextualized representations of all subwords in a word; using only the first fragment can lose lexical or morphological information. For an illustrative split such as `in` + `utile`, the mean includes both pieces. GePpeTto uses the official `GPT2ForTokenClassification` backbone and head, with this shared word pooling added before classification. Its attention remains causal.


In [ ]:
def get_tokenizer(spec):
    options = {"add_prefix_space": True} if spec["kind"] == "decoder" else {}
    tokenizer = AutoTokenizer.from_pretrained(
        spec["repo_id"], revision=spec["revision"], use_fast=True,
        cache_dir=str(PROJECT / "data/hf-cache"), **options,
    )
    if spec["kind"] == "decoder":
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer


def encode_sentences(sentences, tokenizer, variant, max_length):
    if not tokenizer.is_fast or tokenizer.padding_side != "right":
        raise ValueError("A fast tokenizer with right padding is required for word tagging")
    examples = []
    for index, sentence in enumerate(sentences):
        encoded = tokenizer(transform_tokens(sentence["tokens"], variant), is_split_into_words=True, truncation=False)
        if len(encoded["input_ids"]) > max_length:
            raise ValueError(f"Sentence {sentence['sent_id']} ({index}) exceeds max_length={max_length}; no words were truncated")
        ids = [-1 if word is None else word for word in encoded.word_ids()]
        if set(ids) - {-1} != set(range(len(sentence["tokens"]))):
            raise ValueError(f"Incomplete subword coverage: {sentence['sent_id']}")
        examples.append({**dict(encoded), "word_ids": ids, "labels": [LABEL_IDS[tag] for tag in sentence["upos"]]})
    return examples

class WordCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, rows):
        inputs = [{key: value for key, value in row.items() if key not in {"word_ids", "labels"}} for row in rows]
        batch = self.tokenizer.pad(inputs, padding=True, return_tensors="pt")
        width = batch["input_ids"].shape[1]
        batch["word_ids"] = torch.tensor([row["word_ids"] + [-1] * (width - len(row["word_ids"])) for row in rows])
        counts = [len(row["labels"]) for row in rows]
        batch["word_counts"] = torch.tensor(counts)
        batch["labels"] = torch.tensor([row["labels"] + [-100] * (max(counts) - len(row["labels"])) for row in rows])
        return batch

def prepare_data(examples, spec, tokenizer):
    rows = encode_sentences(examples, tokenizer, spec["variant"], spec["max_length"])
    positions = [(s, w) for s, row in enumerate(examples) for w in range(len(row["tokens"]))]
    return Dataset.from_list(rows), positions, {"sentences": len(examples), "words": len(positions),
        "longest_sentence_tokens": max((len(row["input_ids"]) for row in rows), default=0)}

_prepared_system = None

def prepare_system(system):
    global _prepared_system, tokenizer, prepared, model_data, collate
    if _prepared_system == system:
        return
    spec = SYSTEMS[system]
    tokenizer = get_tokenizer(spec)
    assert tokenizer.is_fast
    collate = WordCollator(tokenizer)
    prepared = {split: prepare_data(rows, spec, tokenizer)
                for split, rows in sentences.items()}
    model_data = {split: values[0] for split, values in prepared.items()}
    _prepared_system = system
    print(f"{spec['name']}")
    display(pd.DataFrame([{"Split": split, **values[2]} for split, values in prepared.items()]))



## 5. Metrics


In [ ]:
def score_tags(gold, predicted):
    if len(gold) != len(predicted) or not gold:
        raise ValueError("Gold and predictions must have equal, nonzero length")
    if set(gold) - set(UPOS_TAGS):
        raise ValueError("Unknown gold UPOS tag")
    if set(predicted) - set(UPOS_TAGS):
        raise ValueError("Unexpected prediction label")
    actual, guessed = Counter(gold), Counter(predicted)
    correct = Counter(a for a, b in zip(gold, predicted) if a == b)
    per_tag = []
    for tag in UPOS_TAGS:
        tp, support, count = correct[tag], actual[tag], guessed[tag]
        per_tag.append({
            "tag": tag, "support": support, "tp": tp, "fp": count - tp,
            "fn": support - tp, "precision": tp / count if count else 0.0,
            "recall": tp / support if support else 0.0,
            "f1": 2 * tp / (support + count) if support + count else 0.0,
        })
    present = [row["f1"] for row in per_tag if row["support"]]
    return {
        "upos_accuracy": sum(correct.values()) / len(gold),
        "upos_macro_f1_all": sum(row["f1"] for row in per_tag) / len(UPOS_TAGS),
        "upos_macro_f1_present": sum(present) / len(present),
        "words": len(gold),
    }, per_tag


def evaluate_sentences(sentences, predictions):
    if len(sentences) != len(predictions):
        raise ValueError("Sentence count mismatch")
    gold, guessed, rows = [], [], []
    for sentence_index, (sentence, tags) in enumerate(zip(sentences, predictions)):
        if len(tags) != len(sentence["tokens"]):
            raise ValueError("Word count mismatch: predictions cannot silently omit words")
        gold.extend(sentence["upos"])
        guessed.extend(tags)
        for word_index, (word, actual, predicted) in enumerate(zip(sentence["tokens"], sentence["upos"], tags)):
            rows.append({
                "sentence_index": sentence_index, "sent_id": sentence["sent_id"],
                "word_index": word_index, "word": word,
                "true_label": actual, "predicted_label": predicted,
                "correct": actual == predicted,
            })
    scores, per_tag = score_tags(gold, guessed)
    scores["sentences"] = len(sentences)
    return scores, per_tag, rows

metric_names = ["upos_accuracy", "upos_macro_f1_all", "upos_macro_f1_present"]


def gold_annotations(split):
    return [tag for row in sentences[split] for tag in row["upos"]]


def decode_output(predictions, label_tensor):
    values = predictions[0] if isinstance(predictions, tuple) else predictions
    if values.ndim == 3:
        values = values.argmax(-1)
    if values.shape != label_tensor.shape or values.ndim != 2:
        raise ValueError("Expected one prediction per gold word from Trainer")
    names = UPOS_TAGS
    rows = []
    for row, labels in zip(values, label_tensor):
        ids = row[labels != -100]
        if any(int(i) < 0 or int(i) >= len(names) for i in ids):
            raise ValueError("Unknown predicted UPOS ID")
        rows.append([names[int(i)] for i in ids])
    return rows


def metrics_for(split):
    gold = gold_annotations(split)
    def compute(output):
        predictions = decode_output(output.predictions, output.label_ids)
        valid = output.label_ids != -100
        expected = np.array([LABEL_IDS[tag] for tag in gold])
        assert np.array_equal(output.label_ids[valid], expected), "Gold word order changed"
        scores, _, _ = evaluate_sentences(sentences[split], predictions)
        return scores
    return compute


def reduce_logits(logits, labels):
    logits = logits[0] if isinstance(logits, tuple) else logits
    return logits.argmax(-1)


## 6. Models


In [ ]:
def mean_pool_words(hidden, word_ids, word_counts, attention_mask=None):
    if hidden.ndim != 3 or word_ids.shape != hidden.shape[:2]:
        raise ValueError("Expected hidden[B,S,H] and word_ids[B,S]")
    if word_counts.shape != (hidden.shape[0],) or not bool((word_counts > 0).all()):
        raise ValueError("Every sentence must contain at least one word")
    max_words = int(word_counts.max())
    valid = word_ids >= 0
    if attention_mask is not None:
        valid = valid & attention_mask.bool()
    if bool((valid & (word_ids >= word_counts[:, None])).any()):
        raise ValueError("A subword points outside its sentence's word inventory")
    # Masked special tokens and padding contribute zero to the word sums.
    indices = word_ids.clamp(min=0, max=max_words - 1)
    sums = hidden.new_zeros((hidden.shape[0], max_words, hidden.shape[-1]))
    sums.scatter_add_(1, indices.unsqueeze(-1).expand_as(hidden), hidden * valid.unsqueeze(-1))
    counts = hidden.new_zeros((hidden.shape[0], max_words))
    counts.scatter_add_(1, indices, valid.to(hidden.dtype))
    word_mask = torch.arange(max_words, device=hidden.device)[None, :] < word_counts[:, None]
    if bool((word_mask & (counts == 0)).any()):
        raise ValueError("At least one gold word has no subword representation")
    return sums / counts.clamp(min=1).unsqueeze(-1), word_mask


class EncoderTagger(PreTrainedModel):
    config_class = AutoConfig
    base_model_prefix = "backbone"
    pipeline_version = "upos_mean_pooling_trainer_v1"
    accepts_loss_kwargs = False

    def __init__(self, config, backbone=None):
        self._supports_sdpa = config.model_type == "bert"
        super().__init__(config)
        options = {"add_pooling_layer": False} if config.model_type == "bert" else {}
        self.backbone = backbone if backbone is not None else AutoModel.from_config(config, **options)
        self.dropout = nn.Dropout(config.tagger_dropout)
        self.classifier = nn.Linear(config.hidden_size, len(UPOS_TAGS))
        nn.init.normal_(self.classifier.weight, std=config.initializer_range)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None,
                word_ids=None, word_counts=None, labels=None):
        hidden = self.backbone(input_ids=input_ids, attention_mask=attention_mask,
            token_type_ids=token_type_ids, return_dict=True).last_hidden_state
        words, mask = mean_pool_words(hidden, word_ids, word_counts, attention_mask)
        logits = self.classifier(self.dropout(words))
        loss = None
        if labels is not None:
            if labels.shape != mask.shape or not torch.equal(labels != -100, mask):
                raise ValueError("Labels must cover every pooled word exactly once")
            loss = nn.functional.cross_entropy(logits.reshape(-1, len(UPOS_TAGS)), labels.reshape(-1), ignore_index=-100)
            if not bool(torch.isfinite(loss)):
                raise FloatingPointError("Non-finite loss")
        return TokenClassifierOutput(loss=loss, logits=logits)


class GPT2WordTagger(GPT2ForTokenClassification):
    # Keep the official GPT-2 backbone, dropout and classification head;
    # aggregate subword hidden states before applying the word-level head.
    pipeline_version = "gpt2_upos_mean_pooling_v1"
    accepts_loss_kwargs = False

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None,
                position_ids=None, word_ids=None, word_counts=None, labels=None):
        output = self.transformer(input_ids=input_ids, attention_mask=attention_mask,
            token_type_ids=token_type_ids, position_ids=position_ids, use_cache=False, return_dict=True)
        words, mask = mean_pool_words(output.last_hidden_state, word_ids, word_counts, attention_mask)
        logits = self.classifier(self.dropout(words))
        loss = None
        if labels is not None:
            if labels.shape != mask.shape or not torch.equal(labels != -100, mask):
                raise ValueError("Labels must cover every pooled word exactly once")
            loss = nn.functional.cross_entropy(logits.reshape(-1, self.num_labels), labels.reshape(-1), ignore_index=-100)
            if not bool(torch.isfinite(loss)):
                raise FloatingPointError("Non-finite loss")
        return TokenClassifierOutput(loss=loss, logits=logits)


def make_model(system, checkpoint: str | Path | None = None):
    spec = SYSTEMS[system]
    if spec["kind"] == "encoder":
        if checkpoint is not None:
            model = EncoderTagger.from_pretrained(checkpoint)
            assert model.config.pipeline_version == EncoderTagger.pipeline_version
            assert model.config.id2label == dict(enumerate(UPOS_TAGS))
            return model
        config = AutoConfig.from_pretrained(spec["repo_id"], revision=spec["revision"],
                                           cache_dir=str(PROJECT / "data/hf-cache"))
        config.tagger_dropout = spec["dropout"]
        config.pipeline_version = EncoderTagger.pipeline_version
        config.id2label, config.label2id = dict(enumerate(UPOS_TAGS)), LABEL_IDS
        options = {"add_pooling_layer": False} if config.model_type == "bert" else {}
        backbone = AutoModel.from_pretrained(spec["repo_id"], revision=spec["revision"], config=config,
                                            cache_dir=str(PROJECT / "data/hf-cache"), **options)
        if spec["max_length"] > config.max_position_embeddings:
            raise ValueError("Configured sequence length exceeds encoder positional embeddings")
        model = EncoderTagger(config, backbone)
        assert all(parameter.requires_grad for parameter in model.parameters())
        return model
    if spec["kind"] != "decoder":
        raise ValueError(f"Unknown model kind: {spec['kind']}")
    assert spec["tagger_version"] == GPT2WordTagger.pipeline_version, "Restart the kernel and run all cells in order"
    if checkpoint is not None:
        model = GPT2WordTagger.from_pretrained(checkpoint)
        assert model.config.pipeline_version == GPT2WordTagger.pipeline_version
        assert model.config.id2label == dict(enumerate(UPOS_TAGS))
        return model
    config = AutoConfig.from_pretrained(spec["repo_id"], revision=spec["revision"],
                                       cache_dir=str(PROJECT / "data/hf-cache"))
    config.id2label, config.label2id = dict(enumerate(UPOS_TAGS)), LABEL_IDS
    config.classifier_dropout = spec["dropout"]
    config.pipeline_version = GPT2WordTagger.pipeline_version
    for name in ("bos_token_id", "pad_token_id", "eos_token_id"):
        setattr(config, name, getattr(tokenizer, name))
    config.use_cache = False
    if spec["max_length"] > config.max_position_embeddings:
        raise ValueError("Configured sequence length exceeds model context")
    model = GPT2WordTagger.from_pretrained(spec["repo_id"], revision=spec["revision"], config=config,
                                         cache_dir=str(PROJECT / "data/hf-cache"))
    assert model.config.vocab_size == len(tokenizer), "Tokenizer vocabulary must remain unchanged"
    assert all(parameter.requires_grad for parameter in model.parameters())
    return model


def make_trainer(system, model, run_dir, seed, training=False):
    spec = SYSTEMS[system]
    options: dict[str, Any] = dict(num_train_epochs=spec["epochs"], learning_rate=spec["learning_rate"],
        per_device_train_batch_size=spec["batch_size"], gradient_accumulation_steps=spec["gradient_accumulation"],
        weight_decay=spec["weight_decay"], warmup_ratio=spec["warmup_ratio"],
        eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model=EXPERIMENT["selection_metric"], greater_is_better=True, save_total_limit=1,
        gradient_checkpointing=spec.get("gradient_checkpointing", False),
        gradient_checkpointing_kwargs={"use_reentrant": False}) if training else {}
    cuda = torch.cuda.is_available()
    bf16 = cuda and torch.cuda.is_bf16_supported()
    args = TrainingArguments(output_dir=str(run_dir / ("checkpoints" if training else "evaluation")),
        per_device_eval_batch_size=spec["eval_batch_size"], seed=seed, data_seed=seed,
        fp16=cuda and not bf16, bf16=bf16, optim="adamw_torch", report_to="none", label_names=["labels"],
        remove_unused_columns=False, dataloader_num_workers=0, disable_tqdm=False, **options)
    return Trainer(model=model, args=args, processing_class=tokenizer, data_collator=collate,
        train_dataset=cast(TorchDataset[Any], model_data["train"]) if training else None,
        eval_dataset=cast(TorchDataset[Any], model_data["dev"]) if training else None,
        compute_metrics=metrics_for("dev" if training else "test"),
        preprocess_logits_for_metrics=reduce_logits)


def release_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 7. Training


In [ ]:
def write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def write_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if fieldnames is None:
        fieldnames = list(rows[0]) if rows else []
    with Path(path).open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def package_versions():
    versions = {"python": platform.python_version()}
    for name in ("torch", "transformers", "datasets", "huggingface-hub", "numpy", "accelerate"):
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

def run_identity(config, manifest, model_key, variant, seed):
    return {"configuration": config, "model": model_key, "variant": variant,
            "seed": seed, "data_manifest": manifest}

def model_experiment(config, model_key):
    orchestration = {"models", "active_systems", "seeds", "variants"}
    core = {key: value for key, value in config.items() if key not in orchestration}
    model = {key: value for key, value in config["models"][model_key].items() if key != "seeds"}
    return {**core, "models": {model_key: model}}


def verify_completed_run(run_dir, expected_identity):
    marker = json.loads((run_dir / "complete.json").read_text(encoding="utf-8"))
    if marker["experiment_sha256"] != sha256(run_dir / "experiment.json") or marker["test_results_sha256"] != sha256(run_dir / "test_results.json"):
        raise ValueError(f"Completed run checksum mismatch: {run_dir}")
    saved = json.loads((run_dir / "experiment.json").read_text(encoding="utf-8"))
    model_key = expected_identity["model"]
    actual = {**saved, "configuration": model_experiment(saved["configuration"], model_key)}
    expected = {**expected_identity, "configuration": model_experiment(expected_identity["configuration"], model_key)}
    if actual != expected:
        raise ValueError(f"Run configuration or corpus differs: {run_dir}")


def register_experiment(root, settings):
    path = root / "experiment.json"
    if path.exists():
        saved = json.loads(path.read_text(encoding="utf-8"))
        plan_keys = {"models", "active_systems", "seeds", "variants"}
        old_core = {key: value for key, value in saved.items() if key not in plan_keys}
        new_core = {key: value for key, value in settings.items() if key not in plan_keys}
        assert old_core == new_core, f"Configuration mismatch: {root}"
        for model_key, spec in settings["models"].items():
            if not any(p.is_file() for p in (root / model_key).rglob("*")):
                continue
            previous = saved["models"].get(model_key)
            assert previous is not None, f"Missing provenance for saved model: {model_key}"
            previous = {key: value for key, value in previous.items() if key != "seeds"}
            requested = {key: value for key, value in spec.items() if key != "seeds"}
            assert previous == requested, f"Configuration mismatch for saved model: {model_key}"
        if saved != settings:
            write_json(path, settings)
        return settings
    root.mkdir(parents=True, exist_ok=True)
    write_json(path, settings)
    return settings


def run_directory(system, seed):
    spec = SYSTEMS[system]
    return EXPERIMENT_DIR / spec["model"] / spec["variant"] / f"seed_{seed}"


def run_plan(config, project):
    return [{"model": SYSTEMS[system]["model"], "variant": SYSTEMS[system]["variant"], "seed": seed,
             "directory": str(Path(config["output_dir"]) / SYSTEMS[system]["model"] / SYSTEMS[system]["variant"] / f"seed_{seed}")}
            for system in config["active_systems"]
            for seed in config["models"][SYSTEMS[system]["model"]].get("seeds", config["seeds"])]


def model_weights_path(system, directory):
    return directory / "model.safetensors"


def saved_run_info(system, seed):
    run_dir = run_directory(system, seed)
    info = json.loads((run_dir / "training_info.json").read_text(encoding="utf-8"))
    model_key = SYSTEMS[system]["model"]
    assert model_experiment(info["experiment"], model_key) == model_experiment(experiment, model_key)
    assert info["system"] == system and info["seed"] == seed
    assert info["training_args"]["metric_for_best_model"] == EXPERIMENT["selection_metric"]
    assert model_weights_path(system, run_dir / "best_model").exists(), "Missing saved model"
    return info


def train_run(system, seed):
    spec = SYSTEMS[system]
    run_dir = run_directory(system, seed)
    info_path, model_path = run_dir / "training_info.json", run_dir / "best_model"
    if info_path.exists() != model_weights_path(system, model_path).exists():
        raise RuntimeError(f"Incomplete saved run: {run_dir}. Restore its files or choose a new output directory.")
    if info_path.exists():
        saved_run_info(system, seed)
        print(f"Reusing {system}, seed {seed}")
        return
    set_seed(seed)
    prepare_system(system)
    model = make_model(system)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    trainer = make_trainer(system, model, run_dir, seed, training=True)
    assert trainer.args.metric_for_best_model == EXPERIMENT["selection_metric"]
    print(f"{spec['name']} | seed {seed}; trainable parameters: {trainable:,}; selection: {EXPERIMENT['selection_metric']}")
    started_at, start = datetime.now(timezone.utc).isoformat(), time.perf_counter()
    trainer.train()
    training_seconds = time.perf_counter() - start
    best_metric = trainer.state.best_metric
    if best_metric is None or not np.isfinite(best_metric):
        raise RuntimeError("Training finished without a finite development selection score")
    trainer.save_model(str(model_path))
    pd.DataFrame(trainer.state.log_history).to_csv(run_dir / "training_history.csv", index=False)
    info = {"system": system, "model": spec["model"], "variant": spec["variant"], "seed": seed,
        "experiment": experiment, "started_at_utc": started_at, "packages": package_versions(),
        "parameters_trainable": trainable, "selection_metric": EXPERIMENT["selection_metric"],
        "best_dev_score": float(best_metric), "best_checkpoint": trainer.state.best_model_checkpoint,
        "source_checkpoint": spec["repo_id"], "source_revision": spec["revision"],
        "training_seconds": training_seconds, "training_args": trainer.args.to_dict()}
    write_json(info_path, info)
    write_json(run_dir / "experiment.json", run_identity(EXPERIMENT, data_manifest, spec["model"], spec["variant"], seed))
    del trainer, model
    release_memory()


if EncoderTagger.pipeline_version != EXPERIMENT["schema"]:
    raise RuntimeError("Outdated model definition: restart the kernel and run all cells in order")
experiment = register_experiment(EXPERIMENT_DIR, {**EXPERIMENT, "data_manifest": data_manifest})
completed = {(system, seed): all(path.exists() for path in
    [run_directory(system, seed) / "training_info.json", model_weights_path(system, run_directory(system, seed) / "best_model")])
    for system in ACTIVE_SYSTEMS for seed in SYSTEMS[system].get("seeds", SEEDS)}
plan_path = EXPERIMENT_DIR / "run_plan.json"
write_json(plan_path, {"active_systems": ACTIVE_SYSTEMS,
    "model_seeds": {model: spec.get("seeds", SEEDS) for model, spec in MODELS.items()},
    "saved_runs": sum(completed.values()), "training_runs": sum(not done for done in completed.values())})
print(f"{sum(completed.values())} saved runs; {sum(not done for done in completed.values())} missing runs.")
for system in ACTIVE_SYSTEMS:
    for seed in SYSTEMS[system].get("seeds", SEEDS):
        train_run(system, seed)


## 8. Evaluation


In [ ]:
for seed in SEEDS:
    for system in ACTIVE_SYSTEMS:
        if seed not in SYSTEMS[system].get("seeds", SEEDS):
            continue
        assert (run_directory(system, seed) / "training_info.json").exists(), "Complete all configured training runs first"

run_results, prediction_frames = [], {}
for seed in SEEDS:
    for system in ACTIVE_SYSTEMS:
        if seed not in SYSTEMS[system].get("seeds", SEEDS):
            continue
        run_dir, spec = run_directory(system, seed), SYSTEMS[system]
        info = saved_run_info(system, seed)
        prepare_system(system)
        model = make_model(system, run_dir / "best_model")
        trainer = make_trainer(system, model, run_dir, seed)
        output = trainer.predict(cast(TorchDataset[Any], model_data["test"]))
        assert output.metrics is not None, "Missing test metrics"
        predictions = decode_output(output.predictions, output.label_ids)
        scores, per_tag, rows = evaluate_sentences(sentences["test"], predictions)
        for key in metric_names:
            assert math.isclose(scores[key], output.metrics[f"test_{key}"])
        inference_seconds = float(output.metrics["test_runtime"])
        scores.update(test_inference_seconds=inference_seconds,
                      test_words_per_second=scores["words"] / inference_seconds)
        frame = pd.DataFrame(rows)
        frame.to_csv(run_dir / "predictions.csv", index=False)
        frame[~frame.correct].to_csv(run_dir / "errors.csv", index=False)
        pd.DataFrame(per_tag).to_csv(run_dir / "per_tag_metrics.csv", index=False)
        write_json(run_dir / "test_results.json", scores)
        write_json(run_dir / "complete.json", {"completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "experiment_sha256": sha256(run_dir / "experiment.json"), "test_results_sha256": sha256(run_dir / "test_results.json")})
        run_results.append({"system": system, "model": spec["model"], "variant": spec["variant"], "seed": seed,
                            **scores, "training_seconds": info["training_seconds"], "parameters_trainable": info["parameters_trainable"]})
        prediction_frames[(seed, system)] = frame
        del trainer, model, output
        release_memory()
results = pd.DataFrame(run_results)
results.to_csv(EXPERIMENT_DIR / "results.csv", index=False)
display(results[["system", "seed", *metric_names]].round(5))


## 9. Comparison

In [ ]:
def summarize(config, project):
    completed = [run for run in run_plan(config, project) if (Path(project) / run["directory"] / "complete.json").exists()]
    if not completed:
        raise RuntimeError("No completed runs: summary tables cannot be generated before training")
    _, manifest = load_corpus(config, project)
    rows = []
    for run in completed:
        directory = Path(project) / run["directory"]
        verify_completed_run(directory, run_identity(config, manifest, run["model"], run["variant"], run["seed"]))
        scores = json.loads((directory / "test_results.json").read_text(encoding="utf-8"))
        info = json.loads((directory / "training_info.json").read_text(encoding="utf-8"))
        rows.append({"system": f"{run['model']}_{run['variant']}",
                     "model": run["model"], "variant": run["variant"], "seed": run["seed"],
                     **scores, "training_seconds": info["training_seconds"], "parameters_trainable": info["parameters_trainable"]})
    root = Path(project) / config["output_dir"]
    fields = list(dict.fromkeys(key for row in rows for key in row))
    write_csv(root / "results.csv", rows, fields)
    metric_keys = ("upos_accuracy", "upos_macro_f1_all", "upos_macro_f1_present", "training_seconds", "test_words_per_second")
    summary, changes = [], []
    for model in config["models"]:
        for variant in config["variants"]:
            group = [row for row in rows if row["model"] == model and row["variant"] == variant]
            if not group:
                continue
            entry = {"model": model, "variant": variant, "completed_seeds": len(group),
                     "planned_seeds": len(config["models"][model].get("seeds", config["seeds"]))}
            for key in metric_keys:
                if any(key not in row for row in group):
                    continue
                values = [row[key] for row in group]
                entry[f"{key}_mean"] = statistics.mean(values)
                entry[f"{key}_std"] = statistics.stdev(values) if len(values) > 1 else ""
            summary.append(entry)
        for seed in config["models"][model].get("seeds", config["seeds"]):
            pair = {row["variant"]: row for row in rows if row["model"] == model and row["seed"] == seed}
            if set(pair) != set(config["variants"]):
                continue
            changes.append({"model": model, "seed": seed, **{
                f"{key}_delta_pp": 100 * (pair["lowercase"][key] - pair["original"][key])
                for key in metric_keys[:3]}})
    write_csv(root / "comparison_mean_std.csv", summary, list(dict.fromkeys(key for row in summary for key in row)))
    write_csv(root / "ablation_paired_deltas.csv", changes,
              ["model", "seed", *[f"{key}_delta_pp" for key in metric_keys[:3]]])
    ablation_summary = []
    for model in config["models"]:
        group = [row for row in changes if row["model"] == model]
        if not group:
            continue
        entry = {"model": model, "paired_seeds": len(group),
                 "planned_seeds": len(config["models"][model].get("seeds", config["seeds"]))}
        for key in metric_keys[:3]:
            values = [row[f"{key}_delta_pp"] for row in group]
            entry[f"{key}_delta_pp_mean"] = statistics.mean(values)
            entry[f"{key}_delta_pp_std"] = statistics.stdev(values) if len(values) > 1 else ""
        ablation_summary.append(entry)
    write_csv(root / "ablation_mean_std.csv", ablation_summary,
              ["model", "paired_seeds", "planned_seeds", *[f"{key}_delta_pp_{stat}" for key in metric_keys[:3] for stat in ("mean", "std")]])
    write_json(root / "summary_status.json", {"completed_runs": len(rows), "planned_runs": len(run_plan(config, project)),
                                              "complete": len(rows) == len(run_plan(config, project)),
                                              "delta_definition": "lowercase minus original, percentage points"})
    return {"completed_runs": len(rows), "planned_runs": len(run_plan(config, project))}


summarize(EXPERIMENT, PROJECT)
summary = results.groupby("system")[metric_names].agg(["mean", "std"]).reindex(ACTIVE_SYSTEMS) * 100
print("Mean and sample standard deviation (%), across three seeds for every model.")
display(summary.round(3))
comparison = results.groupby("system")[metric_names].mean().reindex(ACTIVE_SYSTEMS).reset_index()
comparison["system"] = comparison["system"].map({key: value["name"] for key, value in SYSTEMS.items()})
comparison.to_csv(EXPERIMENT_DIR / "comparison.csv", index=False)
display(comparison.set_index("system").mul(100).round(3))
paired = results.pivot(index="seed", columns="system", values=metric_names).reindex(columns=ACTIVE_SYSTEMS, level="system") * 100
paired.to_csv(EXPERIMENT_DIR / "paired_metrics.csv")
display(paired.round(3))
display(pd.read_csv(EXPERIMENT_DIR / "ablation_paired_deltas.csv").round(3))


## 10. Errors


In [ ]:
error_rows = []
example_rows = []
for system in ACTIVE_SYSTEMS:
    counts = Counter()
    for seed in SYSTEMS[system].get("seeds", SEEDS):
        frame = prediction_frames[(seed, system)]
        wrong = frame[frame.true_label != frame.predicted_label]
        counts.update(zip(wrong.true_label, wrong.predicted_label))
        for row in wrong.head(3).to_dict("records"):
            words = sentences["test"][row["sentence_index"]]["tokens"]
            example_rows.append({"system": system, "seed": seed, **row, "sentence": " ".join(words)})
    error_rows.extend({"system": system, "true_tag": a, "predicted_tag": b, "errors": count}
                      for (a, b), count in counts.most_common(3))
error_table = pd.DataFrame(error_rows)
print("Top three UPOS confusions per system, aggregated across seeds.")
display(error_table)
error_table.to_csv(EXPERIMENT_DIR / "error_summary.csv", index=False)
error_examples = pd.DataFrame(example_rows)
display(error_examples)
error_examples.to_csv(EXPERIMENT_DIR / "error_examples.csv", index=False)
